In [18]:
import lightgbm as lgb
from tqdm import tqdm
# import matplotlib.pyplot as plt
from sklearn.datasets import load_svmlight_file
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, mean_squared_error
# import onnxmltools
# import fastinference.Loader
# import fastinference
import pprint
import numpy as np
import json
import pandas as pd

In [19]:
bst = lgb.Booster(model_file='model.txt')
# bst = lgb.Booster(model_file='model_quantized.txt')
q_bst = bst.model_from_string(open('model_quantized.txt').read())
bst = lgb.Booster(model_file='model.txt')

In [20]:
# in nested ensemble count all predictions not None or null 
def count_leaves(ensemble):
    # number of leaves equals number of nodes that make predictions, i.e. prediction != None or prediction != null
    count = 0
    for model in ensemble.models:
        for node in model.nodes:
            if node.prediction is not None:
                count += 1
    return count

# replace int64 values with int values for json serialization
def convert_int64(obj):
    if isinstance(obj, dict):
        return {k: convert_int64(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_int64(v) for v in obj]
    elif isinstance(obj, np.int64):
        return int(obj)
    else:
        return obj
    
# quantize all threshold and prediction values to given type
# def quantize(ensemble, leaf_type=np.float16, split_type=np.float16):

#     for model in ensemble.models:
#         for node in model.nodes:
#             if node.prediction is not None:
#                 node.prediction = np.array(node.prediction, dtype=leaf_type).tolist()
#             if node.split is not None:
#                 node.split = np.array(node.split, dtype=split_type).tolist()
#     return ensemble

# counts number of nodes (internal + leaves) in a sklearn ensemble model
def count_nodes(model):
    count = 0
    if isinstance(model, (GradientBoostingClassifier, GradientBoostingRegressor)):
        for estimator in model.estimators_:
            for tree in estimator:
                    count += tree.tree_.node_count
    if isinstance(model, lgb.Booster):
        model_json = model.dump_model()
        for tree in model_json['tree_info']:
            count += tree['num_leaves'] * 2 - 1
    return count

def evaluate_model(model, X_test, y_test, task):
    if(isinstance(model, lgb.Booster)):
        y_pred = model.predict(X_test.toarray())#, predict_disable_shape_check=True)
        if task == "multiclass":
            y_pred = np.argmax(y_pred, axis=1)
    else:
        y_pred = model.predict(X_test)
    if task in "binary":
        test_acc = accuracy_score(y_test, np.where(y_pred > 0.5, 1, 0))
    elif task == "regression":
        test_acc = mean_squared_error(y_test, y_pred)
    elif task == "multiclass":
        test_acc = accuracy_score(y_test, y_pred)
    else:
        raise ValueError(f"Unknown task: {task}")
    return test_acc

def load_data(name):
    """
    Load dataset by name. Supported names: 'mushroom', 'kr-vs-kp', 'breastcancer', 'covtype'.
    Returns (X_train, y_train), (X_test, y_test)"""
    # TODO: adapt to other paths
    return load_svmlight_file('C:/Users/Jan Stenkamp/Documents/Arbeit/Boosted Trees/code/win/LightGBM/experiments/data/{}.train'.format(name)), load_svmlight_file('C:/Users/Jan Stenkamp/Documents/Arbeit/Boosted Trees/code/win/LightGBM/experiments/data/{}.test'.format(name))

def quantize(in_path, out_path, data_type="float16"):
    """
    Quantize numbers in lines starting with 'threshold=' or 'leaf_value='
    to the specified numpy data type, while preserving string length
    by padding with zeros if needed.

    Parameters
    ----------
    in_path : str
        Path to the input text file.
    out_path : str
        Path to the output text file.
    data_type : str
        NumPy data type (e.g., 'float16', 'float32').
    """
    with open(in_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        if line.startswith("threshold=") or line.startswith("leaf_value="):
            key, values_str = line.split("=")
            values = values_str.strip().split()
            converted = []
            for v in values:
                # Convert to desired dtype
                # cover values out of range for int8; recover initial v length 
                if data_type == "int8":
                    if float(v) < -128:
                        v = "-128." + "0" * (len(v) - 5)
                    elif float(v) > 127:
                        v = "127." + "0" * (len(v) - 4)
                qv = np.dtype(data_type).type(float(v))
                # convert to float again to have decimal point in string representation
                qv_str = str(float(qv))

                # Preserve original string length as lightgbm knows number of characters per tree (tree_sizes)
                if len(qv_str) < len(v):
                    qv_str = qv_str + "0" * (len(v) - len(qv_str))
                elif len(qv_str) > len(v):
                    # If quantized string is longer, truncate
                    qv_str = qv_str[:len(v)]

                converted.append(qv_str)

            new_line = f"{key}=" + " ".join(converted) + "\n"
            new_lines.append(new_line)
        else:
            new_lines.append(line)  # ensure space before newline

    with open(out_path, "w") as f:
        f.writelines(new_lines)


In [21]:
data = lgb.Dataset('C:/Users/Jan Stenkamp/Documents/Arbeit/Boosted Trees/code/win/LightGBM/experiments/data/{}.train'.format("covtype_multi"))
test_data = lgb.Dataset('C:/Users/Jan Stenkamp/Documents/Arbeit/Boosted Trees/code/win/LightGBM/experiments/data/{}.test'.format("covtype_multi"))
(X_train, y_train), (X_test, y_test) = load_data("covtype_multi")

In [43]:
datasets={
    "breastcancer": ("breastcancer", "binary", 1),
    "kr-vs-kp": ("kr-vs-kp", "binary", 1),
    "covtype": ("covtype", "binary", 1),
    "mushroom": ("mushroom", "binary", 1),
    "california_housing": ("california_housing", "regression", 1),
    "kin8nm": ("kin8nm", "regression", 1),
    "wine": ("wine", "multiclass", 7),
    "covtype_multi": ("covtype_multi", "multiclass", 7)
}


In [44]:
# check if key exists in datasets dictionary
"covtype" in datasets

True

In [22]:
dataset = lgb.Dataset(X_train.toarray(), label=y_train)

In [35]:
bst = lgb.train({'objective': 'multiclass', 'max_depth': 5, 'num_trees': 5, 'num_classes': 7}, dataset)
bst.save_model('model.txt')

[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002448 se

In [25]:
# skbst = lgb.LGBMClassifier(objective='multiclass', max_depth=3, n_estimators=1)
# skbst.fit(X_train, y_train)

In [26]:
quantize('model.txt', 'model_quantized.txt', data_type="float16")

In [27]:
bst = lgb.Booster(model_file='model.txt')
# bst = lgb.Booster(model_file='model_quantized.txt')
q_bst = bst.model_from_string(open('model_quantized.txt').read())
bst = lgb.Booster(model_file='model.txt')

In [28]:
print(evaluate_model(bst, X_test, y_test, "multiclass"))
print(evaluate_model(q_bst, X_test, y_test, "multiclass"))
# print(evaluate_model(lgb.Booster(model_file='model_quantized.txt'), X_test, y_test, "multiclass"))

0.7332512929958779
0.7331652366978477


In [34]:
bst.best_score

{}

In [42]:
models=["lgbm_quant", "ccp", "xgb", "cegb"] # quantization is integrated into lgbm training 
datasets=["breastcancer", "kr-vs-kp", "covtype", "mushroom", "california_housing", "kin8nm", "wine", "covtype_multi"]
tasks=["binary", "binary", "binary", "binary", "regression", "regression", "multiclass", "multiclass"]
num_classes=[1, 1, 1, 1, 1, 1, 7, 7] # lightgbm also expects 1 for binary classification
trees=[10] # [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 30, 40, 50, 100, 200, 500, 1000]
depths=[5] #, 5, 7]
alpha=[0.0]#, 0.01]#, 0.02, 0.05, 0.1, 0.2]

def train_models(datasets, trees, depths, alpha, task, output_file="results.csv"):
    # create csv file with header
    with open(output_file, "w") as f:
        f.write("model,dataset,max_trees,no_trees,depth,alpha,train_loss,test_accuracy,sk_nodes\n")
    for m in models:
        print(m)
        for d_i, d in enumerate(datasets):
            print(d)
            (X_train, y_train), (X_test, y_test) = load_data(d)
            for t in tqdm(trees):
                for dep in depths:
                    for a in alpha:
                        if m == "lgbm_quant":
                            if a != 0.0:
                                continue
                            dataset = lgb.Dataset(X_train, label=y_train)
                            model = lgb.train({'objective': tasks[d_i], 'max_depth': dep, 'num_trees': t, 'num_classes': num_classes[d_i]}, dataset)
                            estimators = model.num_trees()
                            train_score = evaluate_model(model, X_train, y_train, task[d_i])
                            nodes = count_nodes(model)
                            # already save results to enable quantization step
                            test_acc = evaluate_model(model, X_test, y_test, task[d_i])
                            with open(output_file, "a") as f:
                                name = "lgbm_base"
                                f.write(f"{name},{d},{t},{estimators},{dep},{a},{train_score},{test_acc},{nodes}\n")

                            # quantize
                            model.save_model('model.txt')
                            quantize('model.txt', 'model_quantized.txt', data_type="float16")
                            model.model_from_string(open('model_quantized.txt').read())                    
                            train_score = None # not available after quantization
                            test_acc = evaluate_model(model, X_test, y_test, task[d_i])                            
                            # TODO: ? actually #nodes and #trees stay the same after quantization

                        elif m == "ccp":
                            if task[d_i] == "regression":
                                model = GradientBoostingRegressor(n_estimators=t, max_depth=dep, ccp_alpha=a)
                            else:
                                model = GradientBoostingClassifier(n_estimators=t, max_depth=dep, ccp_alpha=a)
                            model.fit(X_train, y_train)
                            nodes = count_nodes(model)
                            train_score = model.train_score_
                            estimators = len(model.estimators_)
                        else:
                            # TODO: implement other models
                            return

                        test_acc = evaluate_model(model, X_test, y_test, task[d_i])
                        
                        # write in new line of csv file
                        with open(output_file, "a") as f:
                            f.write(f"{m},{d},{t},{estimators},{dep},{a},{train_score},{test_acc},{nodes}\n")


train_models(datasets, trees, depths, alpha, tasks, output_file="results.csv")

lgbm_quant
breastcancer


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 20.36it/s]


[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Info] Number of positive: 169, number of negative: 286
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000439 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4548
[LightGBM] [Info] Number of data points in the train set: 455, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: p

100%|██████████| 1/1 [00:00<00:00, 21.29it/s]

...

100%|██████████| 1/1 [00:00<00:00,  1.05it/s]


covtype_multi


100%|██████████| 1/1 [02:03<00:00, 123.17s/it]


xgb
breastcancer


  0%|          | 0/1 [00:00<?, ?it/s]
